In [ ]:
!pip install google-generativeai moviepy replicate

import os
import google.generativeai as palm
import replicate

# Configure API keys
palm.configure(api_key=os.environ["GOOGLE_API_KEY"])
replicate_api_token = os.environ["REPLICATE_API_TOKEN"]

# Function to generate text content using Google PaLM API
def generate_content(content_type, topic, tone, audience):
    """Generates text content using Google PaLM API based on user input."""
    prompt = (
        f"Create a {content_type} about the following topic: '{topic}'\n"
        f"Tone: {tone}\n"
        f"Target audience: {audience}"
    )
    completion = palm.generate_text(
        model='models/text-bison-001',
        prompt=prompt,
        temperature=0.7,
        max_output_tokens=500
    )
    return completion.result

# Function to generate an image using Replicate API
def generate_image(prompt):
    """Generates an image using the Replicate API based on a text prompt."""
    client = replicate.Client(api_token=replicate_api_token)
    # List available versions of the stable-diffusion model
    model = client.models.get("stability-ai/stable-diffusion")
    versions = model.versions.list()
    if versions:
        version = versions[0]  # Use the first available version
        output = client.run(
            f"{model.owner}/{model.name}:{version.id}",
            input={"prompt": prompt}
        )
        return output[0]  # Return the generated image URL
    else:
        raise ValueError("No available versions for the specified model.")

# Function to generate a video using Replicate API
def generate_video(prompt):
    """Generates a video using the Replicate API based on a text prompt."""
    client = replicate.Client(api_token=replicate_api_token)
    # List available versions of the zeroscope-v2-xl model
    model = client.models.get("anotherjesse/zeroscope-v2-xl")
    versions = model.versions.list()
    if versions:
        version = versions[0]  # Use the first available version
        output = client.run(
            f"{model.owner}/{model.name}:{version.id}",
            input={"prompt": prompt}
        )
        return output  # Return the generated video URL
    else:
        raise ValueError("No available versions for the specified model.")

# Ensure there are no naming conflicts
try:
    del input  # Delete any redefined 'input' if it exists
except NameError:
    pass

# Get user input
content_type = input("Enter content type (blog post, social media caption, image, video): ")
topic = input("Enter the topic: ")
tone = input("Enter the tone (e.g., informative, persuasive, humorous): ")
audience = input("Enter the target audience: ")

# Generate content based on type
if content_type in ["blog post", "social media caption"]:
    generated_content = generate_content(content_type, topic, tone, audience)
    print("\nGenerated content:\n", generated_content)
elif content_type == "image":
    generated_image_url = generate_image(topic)
    print("\nGenerated image URL:\n", generated_image_url)
elif content_type == "video":
    generated_video_url = generate_video(topic)
    print("\nGenerated video URL:\n", generated_video_url)
else:
    print("Invalid content type.")
